# Week 10: Transport Networks

This notebook covers:
- Downloading street networks with OSMnx
- Calculating travel-time isochrones
- Analysing population coverage

In [ ]:
# Setup (run first)
import sys
if 'google.colab' in sys.modules:
    !pip install geopandas osmnx networkx -q

import geopandas as gpd
import networkx as nx
import osmnx as ox
import matplotlib.pyplot as plt
from pathlib import Path

ox.settings.use_cache = True
print("Ready!")

## 1. Download street network

Change the place name to your study area.

In [ ]:
PLACE = "Melbourne, Victoria, Australia"

G = ox.graph_from_place(PLACE, network_type="walk")

print(f"Nodes: {len(G.nodes)}")
print(f"Edges: {len(G.edges)}")

## 2. Visualise the network

In [ ]:
fig, ax = ox.plot_graph(G, figsize=(10, 10), node_size=0, edge_linewidth=0.3)

## 3. Load facility locations

Or create sample points.

In [ ]:
DATA = Path("../data/processed/week10")

if (DATA / "facilities.geojson").exists():
    facilities = gpd.read_file(DATA / "facilities.geojson").to_crs(4326)
else:
    # Sample facility in Melbourne CBD
    from shapely.geometry import Point
    facilities = gpd.GeoDataFrame(
        {"name": ["Melbourne Central"]},
        geometry=[Point(144.9631, -37.8102)],
        crs="EPSG:4326"
    )

print(f"Facilities: {len(facilities)}")
facilities

## 4. Calculate isochrones

Find all nodes reachable within 5, 10, 15 minutes walking.

In [ ]:
WALK_SPEED = 4.8  # km/h
METERS_PER_MIN = WALK_SPEED * 1000 / 60
TIMES = [5, 10, 15]  # minutes

isochrones = []

for _, fac in facilities.iterrows():
    node = ox.distance.nearest_nodes(G, fac.geometry.x, fac.geometry.y)
    
    for mins in TIMES:
        dist = mins * METERS_PER_MIN
        subgraph = nx.ego_graph(G, node, radius=dist, distance="length")
        
        nodes_gdf = ox.graph_to_gdfs(subgraph, edges=False)
        hull = nodes_gdf.unary_union.convex_hull
        
        isochrones.append({
            "facility": fac.get("name", "Unknown"),
            "minutes": mins,
            "geometry": hull
        })

iso_gdf = gpd.GeoDataFrame(isochrones, crs=G.graph["crs"])
iso_gdf

## 5. Map the isochrones

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

colors = {5: "green", 10: "yellow", 15: "orange"}

for mins in reversed(TIMES):
    iso_gdf[iso_gdf["minutes"] == mins].plot(
        ax=ax, color=colors[mins], alpha=0.4, 
        edgecolor="black", label=f"{mins} min"
    )

facilities.plot(ax=ax, color="red", markersize=100, zorder=5)

ax.legend()
ax.set_title("Walking Time Isochrones")
ax.set_axis_off()
plt.show()

## 6. Export

In [ ]:
output = DATA / "outputs"
output.mkdir(parents=True, exist_ok=True)

iso_gdf.to_file(output / "isochrones.gpkg", driver="GPKG")
print(f"Saved to {output}")

---

**Done!** You've completed network accessibility analysis in Python.